In [2]:
import duckdb
con = duckdb.connect("../data/retail.duckdb", read_only=True)
con.sql("SELECT * FROM raw_transactions LIMIT 10").show()

┌─────────┬────────────┬─────────────────────────────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ invoice │ stock_code │             description             │ quantity │    invoice_date     │ price  │ customer_id │    country     │
│ varchar │  varchar   │               varchar               │  int64   │      timestamp      │ double │   double    │    varchar     │
├─────────┼────────────┼─────────────────────────────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│ 489434  │ 85048      │ 15CM CHRISTMAS GLASS BALL 20 LIGHTS │       12 │ 2009-12-01 07:45:00 │   6.95 │     13085.0 │ United Kingdom │
│ 489434  │ 79323P     │ PINK CHERRY LIGHTS                  │       12 │ 2009-12-01 07:45:00 │   6.75 │     13085.0 │ United Kingdom │
│ 489434  │ 79323W     │  WHITE CHERRY LIGHTS                │       12 │ 2009-12-01 07:45:00 │   6.75 │     13085.0 │ United Kingdom │
│ 489434  │ 22041      │ RECORD FRAME 7" SINGLE 

### Q1: What data range does the data cover? 

In [3]:
con.sql("""SELECT MIN(invoice_date) AS first_date,
        MAX(invoice_date) AS last_date
        FROM raw_transactions""").show()

┌─────────────────────┬─────────────────────┐
│     first_date      │      last_date      │
│      timestamp      │      timestamp      │
├─────────────────────┼─────────────────────┤
│ 2009-12-01 07:45:00 │ 2011-12-09 12:50:00 │
└─────────────────────┴─────────────────────┘



### Q2: How many rows have no customer_id?

In [ ]:
con.sql(""" SELECT COUNT (*) AS customer_id_null
        FROM raw_transactions
        WHERE customer_id IS NULL""").show()

┌──────────────────┐
│ customer_id_null │
│      int64       │
├──────────────────┤
│           243007 │
└──────────────────┘



### Q3: Some invoice numbers start with a letter instead of a number. Which letters, and what do you think they mean?

In [14]:
con.sql("""SELECT LEFT(invoice, 1) AS invoice_first_char,
        FROM raw_transactions
        WHERE NOT regexp_matches(invoice, '^[0-9]')
        GROUP BY LEFT(invoice, 1)
        ORDER BY invoice_first_char""").show()

┌────────────────────┐
│ invoice_first_char │
│      varchar       │
├────────────────────┤
│ A                  │
│ C                  │
└────────────────────┘



In [21]:
con.sql("""SELECT *
        FROM raw_transactions
        WHERE invoice LIKE 'A%'
        LIMIT 10""").show()

┌─────────┬────────────┬─────────────────┬──────────┬─────────────────────┬───────────┬─────────────┬────────────────┐
│ invoice │ stock_code │   description   │ quantity │    invoice_date     │   price   │ customer_id │    country     │
│ varchar │  varchar   │     varchar     │  int64   │      timestamp      │  double   │   double    │    varchar     │
├─────────┼────────────┼─────────────────┼──────────┼─────────────────────┼───────────┼─────────────┼────────────────┤
│ A506401 │ B          │ Adjust bad debt │        1 │ 2010-04-29 13:36:00 │ -53594.36 │        NULL │ United Kingdom │
│ A516228 │ B          │ Adjust bad debt │        1 │ 2010-07-19 11:24:00 │ -44031.79 │        NULL │ United Kingdom │
│ A528059 │ B          │ Adjust bad debt │        1 │ 2010-10-20 12:04:00 │ -38925.87 │        NULL │ United Kingdom │
│ A563185 │ B          │ Adjust bad debt │        1 │ 2011-08-12 14:50:00 │  11062.06 │        NULL │ United Kingdom │
│ A563186 │ B          │ Adjust bad debt │      

#### Q3: found 2 letters: A with 6 rows and C with 10 rows. Letter A is adjust bad debt based on description that's why it's letter A. and C is for cancellation based on the website. 
#### Decision: for sales analysis, i will exclude these because some columns are negative.